# bounce2D x EB-JEPA — full pipeline on Colab

Probing whether an AC-Video-JEPA latent encodes a **conserved invariant** (kinetic energy) — see the repo [README](https://github.com/Mantra20/eb_jepa) and `docs/bounce2d_results.md` for the full story.

Runtime > **A100 GPU** recommended (~4 h for the 300-epoch main run).

1. **CELL 1 (SETUP)** — run ONCE per session: clones the repo, installs deps, mounts Drive.
2. **CELL 2 (CONFIG)** — the **only** place to edit per experiment: `EXP_NAME`, `SIM_COEFF`, `SPEED_*`, `N_BASE`, `EPOCHS`.
3. **CELL 3 (TRAIN)** — checkpoints stream to Drive every 10 epochs; keep the tab alive.
4. **CELL 4 (PROBES + SWEEP)** — full probing suite + checkpoint sweep + figures.
5. **CELL 5 (TRIVIALITY TEST)** — trajectory-level "is E encoded beyond v?" test.

The two published experiments (only `sim_coeff_t` differs):

| | Exp A (main) | Exp B (ablation) |
|---|---|---|
| `SIM_COEFF` | 12 | 0 |
| `EPOCHS` | 300 | 100 |


In [ ]:
# ============================================================
#  CELL 1 — SETUP  (run ONCE per session)
# ============================================================
import os, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip(), "
")

# Drive: checkpoints are written here so a disconnect never loses a run
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/bounce2d_runs", exist_ok=True)

# Clone the project (everything lives in the repo — no files are written by this notebook)
if not os.path.exists("/content/eb_jepa"):
    subprocess.run(["git","clone","--depth","1","https://github.com/Mantra20/eb_jepa.git","/content/eb_jepa"], check=True)
os.chdir("/content/eb_jepa")
subprocess.run(["pip","install","-q","einops","fire","omegaconf","wandb"])

import torch
print("SETUP DONE | torch", torch.__version__, "| cuda", torch.cuda.is_available())


In [ ]:
# ============================================================
#  CELL 2 — CONFIG  (the ONLY place to edit per experiment)
# ============================================================
import os

EXP_NAME  = "main_simt12"   # "main_simt12" (Exp A) | "ablation_simt0" (Exp B)
SIM_COEFF = 12              # 12 = Exp A (main) | 0 = Exp B (ablation)
SPEED_MIN = 0.02            # speed range (energy signal): 0.02-0.10 = x25 spread on v^2
SPEED_MAX = 0.10
N_BASE    = 4000            # training trajectories (~4.3 GB RAM). Raise later if it fits.
EPOCHS    = 300             # 300 for Exp A | 100 for Exp B

# Propagate the speed range to make_data AND training (subprocesses inherit os.environ)
os.environ["BOUNCE_SPEED_MIN"] = str(SPEED_MIN)
os.environ["BOUNCE_SPEED_MAX"] = str(SPEED_MAX)
RUN_DIR = f"/content/drive/MyDrive/bounce2d_runs/{EXP_NAME}_seed1"

# Regenerate the held-out eval set with THIS speed range (consistent with training)
os.chdir("/content/eb_jepa")
if os.path.exists("bounce2d_eval.npz"): os.remove("bounce2d_eval.npz")
os.system("python make_data.py")
print(f"
CONFIG -> {RUN_DIR} | sim={SIM_COEFF} | speed=({SPEED_MIN},{SPEED_MAX}) | n_base={N_BASE} | epochs={EPOCHS}")
print("eval ready:", os.path.exists("/content/eb_jepa/bounce2d_eval.npz"))


In [ ]:
# ============================================================
#  CELL 3 — TRAIN  (~4 h for 300 epochs; checkpoints stream to Drive)
#  Keep the tab ACTIVE (come back and click every ~30 min). Do NOT restart the session.
# ============================================================
import subprocess, sys
cmd = [sys.executable, "scripts/train_bounce2d.py",
       "--exp-name", EXP_NAME,
       "--sim-coeff", str(SIM_COEFF),
       "--speed-min", str(SPEED_MIN), "--speed-max", str(SPEED_MAX),
       "--n-base", str(N_BASE),
       "--epochs", str(EPOCHS),
       "--out-dir", "/content/drive/MyDrive/bounce2d_runs"]
proc = subprocess.Popen(cmd, cwd="/content/eb_jepa",
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end=""); sys.stdout.flush()
proc.wait()
print("
returncode:", proc.returncode, "(-9 = OOM: lower N_BASE in CELL 2)")


In [ ]:
# ============================================================
#  CELL 4 — PROBES + SWEEP
#  For the published Exp A run use --checkpoint e-300.pth.tar (its latest.pth.tar
#  is corrupted) and --max-epoch 300 (the Drive folder holds stale checkpoints).
# ============================================================
import os
CHECKPOINT = "latest.pth.tar"      # or "e-300.pth.tar" for the published Exp A
!cd /content/eb_jepa && python scripts/run_probes.py     --run-dir "{RUN_DIR}" --checkpoint "{CHECKPOINT}" --sweep     --speed-min {SPEED_MIN} --speed-max {SPEED_MAX}

from IPython.display import Image, display
fig = os.path.join(RUN_DIR, "probe_figures.png")
if os.path.exists(fig): display(Image(fig))


In [ ]:
# ============================================================
#  CELL 5 — TRIVIALITY TEST: is E encoded beyond v?
#  (trajectory-level, time-averaged raw latent — complementary to CELL 4,
#   NOT a replicate of it; see docs/probing_energy_triviality.md)
# ============================================================
!cd /content/eb_jepa && python scripts/triviality_test.py     --run-dir "{RUN_DIR}" --checkpoint "{CHECKPOINT}"     --speed-min {SPEED_MIN} --speed-max {SPEED_MAX}


## Reading the training logs

* `eff_rank` — spectral effective rank of the representation covariance. Healthy **> 5**; drifting to 1–2 means representational collapse.
* `temp_var` — temporal variance ratio. Healthy **> 0.1**; going to 0 means temporal collapse (all frames map to the same latent).
* `sim_loss` / `std_loss` / `cov_loss` — the regularizer components, logged per epoch.

If a run collapses, the probes will be spuriously negative — check these numbers before interpreting any probe result.
